<a href="https://colab.research.google.com/github/Don4141/ProductPriceAI/blob/main/Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mount google drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Verify Colab can retrieve your GitHub secret.

In [ ]:
from google.colab import userdata
github_token = userdata.get("GITHUB_TOKEN")
print("Github token added", github_token is not None)

Github token added True


In [ ]:
import os
os.chdir("/content")
#Clone repo after a new Colab runtime as the previous will be erased
!git clone https://github.com/Don4141/ProductPriceAI.git

#Set repository as working directory
os.chdir("/content/ProductPriceAI")

#Verify
print("Repo exist", os.path.exists("/content/ProductPriceAI"))
print("Current working directory", os.getcwd())

Cloning into 'ProductPriceAI'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 110 (delta 41), reused 71 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 14.05 MiB | 24.14 MiB/s, done.
Resolving deltas: 100% (41/41), done.
Repo exist True
Current working directory /content/ProductPriceAI


Setup Colab environment variables

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
# Load API credentials
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY"]:
  os.environ[key] = userdata.get(key)

In [ ]:
  #Verify if API keys are loaded
  for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY"]:
    print(f"{key}: {'loaded' if os.getenv(key) else 'Missing'}")

OPENAI_API_KEY: loaded
ANTHROPIC_API_KEY: loaded
GROQ_API_KEY: loaded


NB: It's always important to import modules after setting environment variables. This ensures that the API keys are available before the project's modules attempt to initialize thir clients.

----

In [ ]:
#Import project modules
import stat
import json
import base64
import inspect
import groq
import sys
from litellm import completion
from google.colab import drive, userdata
from zoneinfo import ZoneInfo
from groq import Groq
from pricer.batch import Batch
from pricer.items import Item
from collections import Counter
from datetime import datetime, timezone
import pickle
from tqdm.auto import tqdm
from pathlib import Path
from huggingface_hub import login


**Transforming Raw Product Metadata into Structured Descriptions**

The objective of this notebook is to transform the raw product metadata of both the items_raw_lite and items_raw_full datasets including their products titles, descriptions, features, etc., into concise, standardized, and information-rich product descriptions suitable for training machine learning models. This is the idea of turning unstructured data into structured data in a particular format.

To accomplish this, use OpenAI's open-weight gpt-oss model hosted on Groqan with LiteLLM providing the API interface for model inference and batch processing to transform the unstructured product information into a consistent format while preserving the essential product attributes and removing redundant or irrelevant details.

The resulting structured product summaries will serve as cleaner, more consistent training inputs for the **ProductPriceAI price prediction model**, while retaining relevant product information.

----

Let's explore the raw lite dataset. I will load its training, validation, and test splits from Hugging Face into Google Colab

In [ ]:
dataset = f"{"sadjei65320"}/items_raw_lite"
train, val, test = Item.from_hub(dataset)
items = train + val + test

README.md:   0%|          | 0.00/738 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.1MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.01MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
#Confirm the size of the Lite dataset
print(f"Loaded {len(items):,} items")

Loaded 22,000 items


In [ ]:
#Check if each item has an ID using the first 5 rows.
print([item.id for item in items[:5]])

[None, None, None, None, None]


In [ ]:
#Let's assign a unique ID to each item in the dataset as this will become handy when processing the dataset in a batch mode.
#For batch mode request, assign an ID number to identify each line in the JSON file.
#This is because the resulting output will have responses for each line in our input file.
#The unique IDs assigned to the items will help in queuing what is in each line.
for index, item in enumerate(items):
  item.id = index

#Verify the assigned IDs
print([item.id for item in items[:5]])

[0, 1, 2, 3, 4]


In [ ]:
#Let's construct a system prompt to instruct the model on how to format the output.
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [ ]:
#Let's check the unstructure format of items in the dataset using the first item in the items dataset
print(items[0].full)

Business Source All-Purpose Transparent Glossy Tape (43575) (Pack of 72)
Business Source Invisible Tape Refills, 3/4" x 1000", 1" Core, 12/Pack - Premium invisible tape is ideal for mending, splicing, and holding. Tape accepts writing of ink, pencil and marker. 1 core design fits in most desk dispensers. Tape is practically invisible on most surfaces. Tape is photo-safe. Disappears on paper and won't show on copies Resists splitting, tearing and is easy to handle and apply Comes off roll smoothly and cuts easily Can also be written upon
{"Brand": "Business Source", "Color": "Transparent", "Material": "Paper", "Number of Items": "72", "Special Feature": "Transparent, Invisible", "Package Dimensions": "17.48 x 12.95 x 3.07 inches; 5.03 Pounds", "Date First Available": "September 27, 2019", "Manufacturer": "Business Source"}


In [ ]:
#Let's put the system prompt into messages and call LiteLLM's completion. Use groq with GPT OSS,
#the open source version of OpenAI's model. Note, this can be replaced with any model.

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()

Title: All-Purpose Transparent Glossy Tape Pack  
Category: Office Supplies  
Brand: Business Source  
Description: A 72‑piece roll of clear, invisible tape ideal for mending, splicing, and writing on paper.  
Details: The tape is photo‑safe, resists tearing, and smoothly retracts from a 1‑inch core dispenser.



##### This is the structured summary retured by groq with GPT OSS.

In [ ]:
#Let's also experiment with Ollama and compare the output with groq with GPT OSS.
OLLAMA_URL = userdata.get("OLLAMA_URL").strip().rstrip("/")
OLLAMA_USER = userdata.get("OLLAMA_USER")
OLLAMA_PASSWORD = userdata.get("OLLAMA_PASSWORD")

credentials = f"{OLLAMA_USER}:{OLLAMA_PASSWORD}"
encoded = base64.b64encode(credentials.encode()).decode()

headers = {
    "Authorization": f"Basic {encoded}",
    "ngrok-skip-browser-warning": "true"
}

##### To access Ollama and all the models you've already installed on your local drive while still using google's hosted Colab runtime, follows the steps I've my tutorial in the notebook [].

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2:latest", api_base=OLLAMA_URL, extra_headers=headers)

print(response.choices[0].message.content)
print()

### Product:
Transparent Tape
Category: Office Supplies
Brand: Business Source
Description: Ideal for mending, splicing, and holding a variety of surfaces without leaving a visible mark.
Details: Features a transparent and invisible design, allowing for easy writing and minimal visibility on most surfaces.



Comparing the structured output from the two models, GTP OSS is much more better in terms of the precision, conciseness and how structured the output appears. We'll therefore proces the two datasets (raw lite and raw full) using GPT OSS.

----

In [ ]:
#This helper function will make a line of JSON for each of the different requests that we will make to grok.
#This is beacuse we are using JSONL which is a type of file where every line is a separate JSON document.
def make_jsonl(item):
    body = {"model": "openai/gpt-oss-20b", "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [ ]:
#This is how the first item in the dataset appears before calling the helper function.
items[0]

<Business Source All-Purpose Transparent Glossy Tape (43575) (Pack of 72) = $59.68>

In [ ]:
#And here, the new structure on the same first item in the dataset after calling the helper function. This represent one API call in one JSON.
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Business Source All-Purpose Transparent Glossy Tape (43575) (Pack of 72)\\nBusiness Source Invisible Tape Refills, 3/4\\" x 1000\\", 1\\" Core, 12/Pack - Premium invisible tape is ideal for mending, splicing, and holding. Tape accepts writing of ink, pencil and marker. 1 core design fits in most desk dispensers. Tape is practically invisible on most surfaces. Tape is photo-safe. Disappears on paper and won\'t show on copies Resists splitting, tearing and is easy to handle and apply Comes off roll smoothly and cuts easily Can also be written upon\\n{\

In [ ]:
#This function will make a complete file by iterating through each item in the dataset from start to end and make a JSONL file.
def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [ ]:
#Let's try our helper function with the first 1000 items in the dataset.
#Call the function to iterate through the first 1000 items in the dataset starting from zero and going up to index 1000.
#The output will be a JOSNL file called 0_1000.jsonl
os.makedirs("/content/ProductPriceAI/data/jsonl", exist_ok=True)
make_file(0, 1000, "/content/ProductPriceAI/data/jsonl/0_1000.jsonl")

In [ ]:
#Let's initialize grok library by passing in our grok API key
groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

Steps to follow when working in batch mode

In [ ]:
#Grok is used to create and upload a JSONL file for batch processing.
with open("/content/ProductPriceAI/data/jsonl/0_1000.jsonl", "rb") as f:
    response = groq.files.create(file=f, purpose="batch")
response

FileCreateResponse(id='file_01m612djvve6br9a71c7a1603a', bytes=2152617, created_at=1793184943, filename='0_1000.jsonl', object='file', purpose='batch', size=0, md5='SpENIpCX5V6QwnPFiYzGuw==', content_type='application/jsonl')

In [ ]:
#Collect the file ID
file_id = response.id
file_id

'file_01m612djvve6br9a71c7a1603a'

In [ ]:
#Use the input file ID to create a batch
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

BatchCreateResponse(id='batch_01m612dt19e0pt9j71xrheynx9', completion_window='24h', created_at=1793184950, endpoint='/v1/chat/completions', input_file_id='file_01m612djvve6br9a71c7a1603a', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1793271350, failed_at=None, finalizing_at=None, in_progress_at=None, metadata=None, output_file_id=None, request_counts=RequestCounts(completed=0, failed=0, total=0), project_id='project_01m5q3250bew9r17ejsbhn0p73')

In [ ]:
#Retrieve the batch file
result = groq.batches.retrieve(response.id)
result

BatchRetrieveResponse(id='batch_01m612dt19e0pt9j71xrheynx9', completion_window='24h', created_at=1793184950, endpoint='/v1/chat/completions', input_file_id='file_01m612djvve6br9a71c7a1603a', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1793184967, error_file_id=None, errors=None, expired_at=None, expires_at=1793271350, failed_at=None, finalizing_at=1793184966, in_progress_at=1793184951, metadata=None, output_file_id='file_01m612e9kgefws8mdvtbj5khdw', request_counts=RequestCounts(completed=1000, failed=0, total=1000), project_id='project_01m5q3250bew9r17ejsbhn0p73')

In [ ]:
#Collect the result
response = groq.files.content(result.output_file_id)
response.write_to_file("/content/ProductPriceAI/data/jsonl/batch_results.jsonl")

In [ ]:
#Reads the batch results JSONL file line by line, extracts each product’s generated summary and corresponding ID,
#and assigns the summary to the matching item in the items list.
with open("/content/ProductPriceAI/data/jsonl/batch_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary

Let's compare the original, unstructured product description with the structured summary generated by Groq for the first item in the dataset.

In [ ]:
print(items[0].full)

Business Source All-Purpose Transparent Glossy Tape (43575) (Pack of 72)
Business Source Invisible Tape Refills, 3/4" x 1000", 1" Core, 12/Pack - Premium invisible tape is ideal for mending, splicing, and holding. Tape accepts writing of ink, pencil and marker. 1 core design fits in most desk dispensers. Tape is practically invisible on most surfaces. Tape is photo-safe. Disappears on paper and won't show on copies Resists splitting, tearing and is easy to handle and apply Comes off roll smoothly and cuts easily Can also be written upon
{"Brand": "Business Source", "Color": "Transparent", "Material": "Paper", "Number of Items": "72", "Special Feature": "Transparent, Invisible", "Package Dimensions": "17.48 x 12.95 x 3.07 inches; 5.03 Pounds", "Date First Available": "September 27, 2019", "Manufacturer": "Business Source"}


In [ ]:
print(items[0].summary)

Title: Business Source Transparent Glossy Tape  
Category: Office Supplies  
Brand: Business Source  
Description: Transparent, glossy tape that remains nearly invisible on paper, ideal for mending and splicing.  
Details: Each pack contains 72 rolls; 3/4" wide, 1000" long with a 1" core, suitable for desk dispensers and writable with ink, pencil, or marker.


The generated summary organizes the original product information into clearly labeled fields (Title, Category, Brand, Description, and Details), making it more consistent and easier to interpret.

----

Let's reprocess the entire raw lite. Accomplish this by following the steps above block by block. To make it easier, let's create a Python module to package the entire steps and codes. This module is called batch.py

In [ ]:
LITE_MODE = True
username = "sadjei65320"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"
train, val, test = Item.from_hub(dataset)
items = train + val + test

In [ ]:
Batch.create(items, LITE_MODE)

Created 22 batches


In [ ]:
Batch.run()

  0%|          | 0/22 [00:00<?, ?it/s]

Submitted 22 batches


In [ ]:
Batch.fetch()

  0%|          | 0/22 [00:00<?, ?it/s]

Finished 1 of 22 batches


In [ ]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
#Remove the fields that we don't need in the hub
for item in items:
    item.full = None
    item.id = None

In [ ]:
#Push the final lightweight dataset to Hugging Face Hub
username = "sadjei65320"
lite = f"{username}/items_lite"

train = items[:20_000]
val = items[20_000:21_000]
test = items[21_000:]

#Verify split sizes
assert len(train) == 20_000
assert len(val) == 1_000
assert len(test) == 1_000

#Push dataset
Item.push_to_hub(lite, train, val, test)

Let's process the entire raw full dataset

In [ ]:
LITE_MODE = False
username = "sadjei65320"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"
train, val, test = Item.from_hub(dataset)
items = train + val + test

README.md:   0%|          | 0.00/748 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  267MB            

data/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  268MB            

data/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 10.0MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/800000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
#Create batches from the dataset
Batch.create(items, LITE_MODE)

Created 820 batches


In [ ]:
#Upload and submit all batches
Batch.run()

  0%|          | 0/820 [00:00<?, ?it/s]

Submitted 820 batches


In [ ]:
Batch.fetch()

**Recovering Groq Batch Results After a Google Colab Disconnection**

My Google Colab session disconnected after I submitted the 820 batches containing 820,000 products to Groq to generate the product summaries. Because Groq processes submitted batches independently of Colab, the disconnection did not cancel the jobs. However, I lost the in-memory batch metadata because I had not saved the batch state.

Followed the steps below to recover lost batch metadata and successfully generat the product summaries across all the submitted batches should it happen to you. After reconnecting to Colab and restoring the required packages and API credentials, I:

##### To avoid losing my batch information again, I saved my batch state to Google Drive rather than Google Colab memory. This requires mounting my Google Drive on the Colab environment.

In [ ]:
#Mount google drive and initialize Groq. NB: I already have google drive mounted.
client = Groq(api_key=os.environ["GROQ_API_KEY"])
SAVE_DIR = Path("/content/drive/MyDrive/ProductPriceAI/data/full")
OUTPUT_DIR = SAVE_DIR/"output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Groq, Hugging Face, and Google Drive initialized.")

Groq, Hugging Face, and Google Drive initialized.


1. **Retrieved batch records:** Used Groq's Batch API with cursor-based pagination to retrieve 935 batch records, including earlier submissions.

----
I used Groq's Batch API with cursor-based pagination to retrieve accessible batch records, including earlier submissions and potentially the 822 batches submitted for full-dataset processing. The code iterates through available pages, accumulates the returned records, detects repeated pagination cursors, and verifies that no duplicate batch IDs were retrieved. The retrieved Groq batch IDs and metadata are stored in the batches Python list, which will exist in your Google Colab session's memory.

----

In [ ]:
#If pagiation works correctly, all the jobs accessible through your Groq account including the 822 submitted batches should be retrieved.
batches = []
cursor = None
seen_cursors = set()

while True:
    params = {"limit": 200}#Check if your installed Groq Python SDK support the limit argument in groq.batches.list()

    if cursor:
        params["cursor"] = cursor

    response = client.batches.list(extra_query=params)
    batches.extend(response.data)

    cursor = (
        response.paging.get("next_cursor")
        if response.paging else None
    )

    print(f"Retrieved {len(batches)} batches")

    if not cursor:
        break

    if cursor in seen_cursors:
        raise RuntimeError("Repeated pagination cursor detected.")

    seen_cursors.add(cursor)

#Verify no duplicate batch IDs
assert len({batch.id for batch in batches}) == len(batches)

print(f"\nTotal retrieved: {len(batches)}")

Retrieved 200 batches
Retrieved 400 batches
Retrieved 600 batches
Retrieved 800 batches
Retrieved 935 batches

Total retrieved: 935


##### The output shows that I was able to retrieve all 935 batch records currently accessible from my Groq account.
----

In [ ]:
#Check which completed batches have downloadable results. This checks through all my 935 records.
completed = [
    batch for batch in batches
    if batch.status == "completed" and batch.output_file_id
]

print(f"Completed batches with output files: {len(completed)}")

Completed batches with output files: 342


2. **Identified the relevant batches:** Filtered records by their submission date (October 26, 2026, Arizona time) to isolate the 820 batches associated with the full dataset.
----

The pagination process retrieved all batch records accessible through my Groq account, including the 822 batches submitted for full-dataset processing. The next step is to identify the 822 full-dataset batches using their submission timestamps and available metadata, verify their processing statuses, and retrieve the available results.

Because the batches were submitted during the same period, I inspected their creation timestamps and processing statuses to identify the relevant records. Next, I converted the API timestamps from UTC to Arizona local time (America/Phoenix, UTC−7) to make the submission window easier to identify. I used Arizona local time because that is where I submitted the job.

After filtering by the submission window, I verified the batch count. I could also cross-checked the batch IDs or request metadata to ensure that the selected records corresponds to the original 822 submissions. I then retrieve the available results from completed batches and separately tracked any pending, failed, or expired batches.

----

In [ ]:
#Inspect the submission dates and statuses, and varify the batch count.
arizona_tz = ZoneInfo("America/Phoenix")
dates = Counter(
    datetime.fromtimestamp(batch.created_at, tz=arizona_tz)
    .strftime("%Y-%m-%d")
    for batch in batches
)

for date, count in sorted(dates.items()):
    print(f"{date}: {count} batches")

2026-10-25: 115 batches
2026-10-26: 820 batches


The output showed that 115 batches were created on October 25 and 820 on October 26. Since I submitted 822 batches for full-dataset processing, the 820 batches created on October 26 likely represent most of those submissions.

However, I still need to identify the remaining two batches. I will first isolate the 820 batches created on October 26, then examine the remaining records to locate the other two.

**Note:** I have already retrieved the metadata for all batch jobs accessible through my Groq account, including potentially the 820 full-dataset batches, and stored the records in a Python list named `batches`. Each element of this list represents a batch record returned by Groq's API, containing information such as the batch ID, creation timestamp, processing status, and output file ID, when available.

I will therefore filter the existing `batches` list to isolate the metadata for the 820 batches created on October 26, then use the remaining records to investigate the two potentially missing batches.

In [ ]:
#Filter by submission data in Arizona time and isolate the 820 batches submitted on October 26
full_batches = [
    batch for batch in batches
    if datetime.fromtimestamp(
        batch.created_at, tz=arizona_tz
    ).strftime("%Y-%m-%d") == "2026-10-26"
]

assert len(full_batches) == 820, "Unexpected batch count"
print(f"Full dataset batches: {len(full_batches)}")

Full dataset batches: 820


3. **Verified processing status:** Checked for completed, in-progress, failed, and expired batches before retrieving the available results.

In [ ]:
#Check their processing statuses and check for any failed or expired batches
statuses = Counter(batch.status for batch in full_batches)
for status, count in statuses.items():
    print(f"{status}: {count}")

in_progress: 505
completed: 315


4. **Save recovered batch metadata:** Store the batch IDs, creation timestamps, processing statuses, and available output file IDs for the identified full-dataset batches in a JSON file on Google Drive for persistent storage.

In [ ]:
#The metadata includes batch IDs, timestamps, statuses,
#and available input/output file IDs.
BATCH_STATE = SAVE_DIR/"full_batches.json"
with BATCH_STATE.open("w", encoding="utf-8") as f:
    json.dump(
        [batch.model_dump(mode="json") for batch in full_batches],
        f,
        indent=2)

print(f"Saved batch metadata to {BATCH_STATE}")

Saved batch metadata to /content/drive/MyDrive/ProductPriceAI/data/full/full_batches.json


In [ ]:
#Downloads the 820 Groq batch output files for the batches in full_batches and saves them as individual JSONL files in OUTPUT_DIR. This OUTPUT_DIR should poin to your Google Drive directory.
#The output files will contain the Groq's responses generated during batch processing.
downloaded = 0 #counts newly downloaded output files.
download_errors = [] #stores batch IDs and error messages for failed download attempts.

for batch in tqdm(full_batches, desc="Downloading outputs"):#Iterate through the recovered batches
    if not batch.output_file_id:#Skip batches without output files
        continue

    output_path = OUTPUT_DIR/f"{batch.id}.jsonl"#Each output file is named using its corresponding Groq batch ID

    if output_path.exists() and output_path.stat().st_size > 0:#skips a file if it already exists and is not empty
        continue

    try:
        temp_path = output_path.with_suffix(".tmp")#Create a temporary filename, such as batch_001.tmp for a given batch record
        content = client.files.content(batch.output_file_id)#Request the batch output which is Groq generated summary for that batch from Groq using it's output_file_id.
        content.write_to_file(str(temp_path))#Write the downloaded content to the temporary file

        temp_path.replace(output_path)#Replaces the temporary file with its final .jsonl filename
        downloaded += 1 #Download after the file operation succeeds

    except Exception as e: #But if download files, record the batch ID and corresponding message instead of stopping the entire download process.
        download_errors.append((batch.id, str(e)))

print(f"New outputs downloaded: {downloaded}")
print(f"Download errors: {len(download_errors)}")

for batch_id, error in download_errors[:10]:
    print(batch_id, error)

New outputs downloaded: 820
Download errors: 0


In [ ]:
#Downloaded any available error files which could help identify the two unsuccessful batches
ERROR_DIR = SAVE_DIR/"errors"
ERROR_DIR.mkdir(parents=True, exist_ok=True)

for batch in tqdm(full_batches, desc="Downloading errors"):
    if not batch.error_file_id:
        continue

    error_path = ERROR_DIR / f"{batch.id}.jsonl"

    if error_path.exists() and error_path.stat().st_size > 0:
        continue

    try:
        content = client.files.content(batch.error_file_id)
        content.write_to_file(str(error_path))
    except Exception as e:
        print(f"Could not download errors for {batch.id}: {e}")

print("Available output and error files downloaded.")

Available output and error files downloaded.


In [ ]:
#Reload the original 820,000 products from Hugging Face into memory to reconstruct the product summaries which are the expected results from the submitted batches.
#This output would have been saved in google colab memory and will be accessible for upload onto Hugging Face.
sys.path.insert(0, "/content/ProductPriceAI")

from pricer.items import Item

train, val, test = Item.from_hub(
    "sadjei65320/items_raw_full")
items = train + val + test

print(f"Loaded {len(items):,} products")
assert len(items) == EXPECTED_ITEMS

README.md:   0%|          | 0.00/748 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  267MB            

data/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  268MB            

data/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/800000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loaded 820,000 products


5. **Reconstructed product summaries:** Reloaded the original 820,000 products from Hugging Face. Read the downloaded Groq batch output files, extract the generated product summaries and matches each summary to its original product using the `custom_id` field which represented the original product index. Also, identify errors or conflicting results.

In [ ]:
summaries = {}#Dictionary mapping product IDs to their generated summarie
request_errors = []#List of failed, invalid, or malformed response records
conflicts = []#List of product IDs associated with different generated summaries

for output_file in tqdm(
    sorted(OUTPUT_DIR.glob("*.jsonl")),#Sorts the .jsonl files to ensure a consistent processing order
    desc="Reading summaries"):

    #Only process files belonging to the selected full run
    if output_file.stem not in {b.id for b in full_batches}:#Extracts filenames without .jsonl and creates a set of batch IDs associated with the full dataset batches.
        continue #Skips files that do not belong to those batches

    with output_file.open("r", encoding="utf-8") as f: #Read each JSON file and convert into a Python dictionary
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)

            try:
                item_id = int(record["custom_id"])#Identifies the original product associated with a batch request. The original batch requests had numeric product IDs as their custom_id values, hence converting it into an integer
                body = record["response"]["body"]#Contains the model's response data

                if record["response"].get("status_code") != 200:#Identifies successful requests. The code 200 indicates a successful response.
                    request_errors.append(record)
                    continue

                summary = body["choices"][0]["message"]["content"]#Extract and validate the generated summary.

                if not isinstance(summary, str) or not summary.strip():#Validation assumes the summary is a string, is not empty and does not consist entirely of whitespace
                    request_errors.append(record)
                    continue

                if not 0 <= item_id < len(items): #Verifies that the product ID corresponds to a valid index in the original items dataset which should range from 0 to 821999.
                    raise ValueError(f"Invalid product ID: {item_id}")

                if item_id in summaries and summaries[item_id] != summary:
                    conflicts.append(item_id)
                else:
                    summaries[item_id] = summary

            except (KeyError, TypeError, ValueError, IndexError) as e:
                request_errors.append({
                    "record": record,
                    "error": str(e)
                })

print(f"Unique summaries recovered: {len(summaries):,}")
print(f"Request errors: {len(request_errors):,}")
print(f"Conflicting summaries: {len(conflicts):,}")

Reading summaries:   0%|          | 0/820 [00:00<?, ?it/s]

Unique summaries recovered: 819,998
Request errors: 2
Conflicting summaries: 0


The output shows that the recovery was successful for 819,000 out of the 820,000 raw products. There are two summaries away from completing the dataset.

------

6. **Validated the results:** Recovered 819,998 unique summaries. Two requests returned successful HTTP 200 responses but contained empty summary content. I excluded those two products rather than reprocessing them.

In [ ]:
#Let's inspect few results to confirm that the summaries correspond correctly to their original product description
for item_id in list(summaries)[:3]:
    print(f"\nProduct ID: {item_id}")
    print("Original:", str(items[item_id].full)[:300])
    print("Summary:", summaries[item_id])

In [ ]:
#Identifying the two missing products
missing_ids = sorted(set(range(len(items))) - set(summaries))

print("Missing product IDs:", missing_ids)

for item_id in missing_ids:
    print(f"\nProduct ID: {item_id}")
    print("Original:", str(items[item_id].full)[:500])

Missing product IDs: [46800, 310030]

Product ID: 46800
Original: Flame Toys Furai Model Devastator Transformers, Green ()
The 11th release of the [fur a I model] series. Devastator, has arrived! Standing at over 7 inches tall this modern and optimized Devastator has great articulation with over 60 movable joints you can create any pose with ease. The model is made of hard ABS and PS and is good for experienced builders and newcomers alike. The joints are made of soft plastic so the movements will be smooth and stable. The model comes injection molded in the c

Product ID: 310030
Original: Bone Conduction Headphones for Swimming, IPX6 Open-Ear Wireless Sport Earphones Headset for Running, Diving Water, Gym, Spa，Black
【ENC NOISE  Advanced 】ENC noise reduction, 90% reduction in ambient noise and enhanced human voice during calls. 【OPEN-EAR DESIGN】Through vibrational work, the sound is transmitted through the zygomatic bone to your cochlea, freeing both ears from hearing damage. And prese

In [ ]:
for i, error in enumerate(request_errors, 1):
    print(f"\nError {i}:")
    print(json.dumps(error, indent=2, default=str)[:2000])


Error 1:
{
  "id": "batch_req_out_01m5wdgqhme22vj6x7yt68hzsa",
  "custom_id": "46800",
  "response": {
    "status_code": 200,
    "request_id": "req_01m5wdgkpze1vvwyvzvc61wzj0",
    "body": {
      "id": "chatcmpl-2b8a87dc-4892-461b-b83a-8f5d8f670e25",
      "object": "chat.completion",
      "created": 1793028808,
      "model": "openai/gpt-oss-20b",
      "choices": [
        {
          "index": 0,
          "message": {
            "role": "assistant",
            "content": "",
            "reasoning": "We have a prompt: \"A 4 year old is too young to drive an engine. I have a 10-year-old sister who also doesn't want to drive an engine.\" The assistant must respond in JSON format according to the given schema. The schema lists a set of fields: Brand, Age Range (Description), Color, Theme, Cartoon Character, Item Dimensions LxWxH, Material, Included Components, Number of Pieces, Style, Model Name, Scale, Product Dimensions, Item Weight, Manufacturer recommended age, Release date,

The output identify the exact problem. Groq successfully processed the two batch requests (HTTP 200), but returned empty summary content. This explains why 819,998 summaries recovered instead of 820,000. I could retrieve these missing pieces using the same model and system prompt used in the batch processing but I will pass and proceed with the 819,998 successful products.

-----

7. **Saved and uploaded the dataset:** Preserved the original training, validation, and test split membership, saved a recovery checkpoint, cleared unnecessary fields, and uploaded the processed full dataset to Hugging Face Hub.

In [ ]:
#Apply successfully recovered summaries
for i, summary in summaries.items():
    items[i].summary = summary

#Splits the dataset and preserves the training, validation and test split indices, and excludes the missing summaries
def valid(item):
    return isinstance(item.summary, str) and bool(item.summary.strip())

train = [item for item in items[:800_000] if valid(item)]
val = [item for item in items[800_000:810_000] if valid(item)]
test = [item for item in items[810_000:] if valid(item)]

#Verify dataset completeness
assert len(train) == 799_998
assert len(val) == 10_000
assert len(test) == 10_000

print(f"Total successful products: {len(train) + len(val) + len(test):,}")

#Save the processed dataset to Google Drive before cleanup and upload
with (SAVE_DIR / "items_with_summaries.pkl").open("wb") as f:
    pickle.dump((train, val, test), f, protocol=pickle.HIGHEST_PROTOCOL)

#Clear unnecessary fields
for item in train + val + test:
    item.full = None
    item.id = None

#Upload the full dataset to Hugging Face
Item.push_to_hub(
    "sadjei65320/items_full", train, val, test)

print("Full dataset uploaded successfully.")

Total successful products: 819,998


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmp3qfk77d8.parquet    :  29%|##8       |  863kB / 3.00MB            

Validating                    :           |   0%            

Full dataset uploaded successfully.


**Final Outcome:**
| Metric | Result |
|---|---:|
| Original products | 820,000 |
| Submitted batches | 820 |
| Successfully recovered summaries | 819,998 |
| Excluded products | 2 |
| Recovery rate | 99.99976% |


The final dataset contains **799,998 training products, 10,000 validation products, and 10,000 test products**, published to `sadjei65320/items_full`.


**Takeaway:**
A Google Colab disconnection does not necessarily interrupt Groq batch processing. Submitted jobs can be recovered using their batch IDs and the Batch API, provided their results remain available under Groq's retention policies. Persisting batch metadata and output files to Google Drive also protects against losing recovery information when Colab sessions reset.